# Tuning-Free Classification and Regression with TabPFN - DSDL Notebook

This notebook implements an on-premises TabPFN model (classification and regression, using the TabPFN-3.5 checkpoints) on top of the Splunk App for Data Science and Deep Learning (DSDL) - formerly known as the Deep Learning Toolkit for Splunk (DLTK). It is based on DSDL's barebone example template; find more background and other algorithm examples in the app itself and on the [DSDL Splunkbase page](https://splunkbase.splunk.com/app/4607/#/details).

**Tested with:** Splunk Enterprise 10.4.1 · AI Toolkit 6.2.0 · DSDL app 5.2.4 · Agentic AI image 5.2.5 (`splunk/mltk-container-ubi-llm-rag:agentic-ai-5.2.5`, Python 3.11) · tabpfn 9.2.0 · checkpoints `tabpfn-v3.5-20260909` (default) and `tabpfn-v3.5-fast-20260909` (fast)

**Quick start**

1. Start a suitable container in DSDL: the **Golden Image CPU** 5.2.5 or later already includes `tabpfn`; for any other image, install it first (see *Requirements* below).
2. Run the **Stage -1** cell once to download the checkpoint(s) you want to use.
3. Save this notebook as `tabpfn_model.ipynb` - the file name is the `algo=` name in SPL.
4. Train from Splunk with `| fit MLTKContainer algo=tabpfn_model <target> from <features> into app:<model_name>` and score with `| apply <model_name>`.

**SPL options** (set after `algo=` in `| fit`, stored with the model and reused by `| apply`):

| Option | Values | Default | Meaning |
|---|---|---|---|
| `checkpoint` | `default`, `fast` | `default` | `default` = TabPFN-3.5 (~876 MB). `fast` = TabPFN-3.5-Fast (~334 MB, up to ~6x faster according to Prior Labs, **alpha**) - a good choice for CPU-only containers. |
| `task` | `classification`, `regression` | `classification` | Both tasks use the same checkpoint file. |
| `ignore_pretraining_limits` | `true`, `false` | `false` | Lifts TabPFN's default CPU limit of 5,000 training rows (slower!). |

**Important:** Every time you save this notebook, only the specifically tagged cells below (`mltkc_import`, `mltkc_init`, `mltkc_fit`, `mltkc_apply`, `mltkc_save`, `mltkc_load`, `mltkc_summary`) are exported into the Python module that actually serves `| fit` / `| apply` / `| summary` calls from Splunk. Any import, constant, or helper function defined in a free/untagged cell - including this notebook's own testing cells above and below - is invisible to production `fit`/`apply` calls, even though it runs fine when you execute it interactively here. **Put anything your model logic depends on into `mltkc_import`, not into a free cell above or below it.** Never use the cell type *Raw* for notes or commands: raw cells can end up in the exported module and break it - use Markdown code fences instead. See the Model Development Guide in the DSDL app for more on this workflow.

**Requirements - Python 3.10 or newer.** TabPFN-3.5 needs `tabpfn` 9.0 or later (3.5 has been the default model since 9.0.0), which requires Python 3.10+. Which image to use depends on the image version:

- **Images 5.2.5 and later, Golden Image CPU:** `tabpfn` and `huggingface_hub` are preinstalled - skip the install below.
- **Images 5.2.5 and later, all other images:** they ship a current Python version, so you only need to install the packages below.
- **Images before 5.2.5:** the golden images run Python 3.9, where `pip install tabpfn` silently falls back to tabpfn 8.x (TabPFN-3). Use DSDL's **Agentic AI** image (Python 3.11) instead and install the packages below.

To install, find the container ID with `docker ps` (match by the `IMAGE` column if more than one container is running), then run these commands from a terminal on the host. Pin the version you tested with: tabpfn 9.1 and 9.2 contain TabPFN-3.5 bug fixes that change predictions, so unpinned installs can give different results.

```
docker exec -u root <container_id> python3 --version        # needs Python 3.10 or later
docker exec -u root <container_id> python3 -m pip install "tabpfn==9.2.0" huggingface_hub
```
If `pip` is missing in your image, run `docker exec -u root <container_id> python3 -m ensurepip --upgrade` first.

If you installed the packages, restart the notebook's kernel afterwards. The install lives in the container's writable layer, so repeat the install if the container is ever recreated (for example after STOP/START in DSDL, a host reboot, or a Docker Desktop restart). The checkpoints downloaded in Stage -1 are stored in `/srv/app/model/data/`. For anything beyond evaluation, use the Golden Image CPU 5.2.5 or later, or build a custom image with `tabpfn` preinstalled, instead of installing it at runtime.

**License:** the TabPFN-3.5 weights (default and fast) are released under the TABPFN-3.5 License v1.0, a non-commercial license. Testing, evaluation, and internal benchmarking are allowed; commercial or production use - including using the model's predictions for business decisions - is not. Read the license (the `LICENSE` file in the model repository) before going beyond evaluation, and contact Prior Labs for a commercial license.

## Stage -1 - one-time setup: download the TabPFN-3.5 checkpoint(s) (run once, not exported)

TabPFN's PyPI package ships no model weights - the checkpoints live in Prior Labs' Hugging Face repository `Prior-Labs/tabpfn_3_5`. Each checkpoint is a single file that serves both classification and regression:

| Name | File | Size | Notes |
|---|---|---|---|
| `default` | `tabpfn-v3.5-20260909.safetensors` | ~876 MB | TabPFN-3.5, best accuracy |
| `fast` | `tabpfn-v3.5-fast-20260909.safetensors` | ~334 MB | TabPFN-3.5-Fast, smaller and faster, **alpha** |

We download them explicitly, once, and load them from a local file afterwards via `model_path=` - so that `fit()`/`apply()` make **no network calls** at runtime. That's what makes this deployment genuinely on-premises rather than just cloud-API-optional. Pick the checkpoint(s) you want in `CHECKPOINTS_TO_DOWNLOAD`; download both if you want to compare them.

At the time of writing the download works without a token; if your network rejects it or Hugging Face asks for authentication, create a *read* token (https://huggingface.co/settings/tokens/new?preset=read-only) and paste it into `HF_TOKEN`. Even without a token, using the weights means accepting the TABPFN-3.5 License v1.0.

If this fails with `SSLCertVerificationError`, the container's network path to huggingface.co is most likely intercepted by a proxy it doesn't trust. In that case, download the file(s) once via a normal browser on the host machine instead, and copy them into the container manually, e.g. `docker cp <file> <container_id>:/srv/app/model/data/`. Then run this cell again: it skips files that are already present and only verifies their checksums.

In [ ]:
# THIS CELL IS NOT EXPORTED - one-time setup, run manually when needed
import hashlib
import os

HF_TOKEN = None  # optional: only needed if the download is rejected or rate-limited
HF_REPO_ID = "Prior-Labs/tabpfn_3_5"
HF_LOCAL_DIR = "/srv/app/model/data/"
CHECKPOINTS_TO_DOWNLOAD = ["default", "fast"]  # pick one or both

# keep file names in sync with CHECKPOINTS in the Stage 0 cell below
CHECKPOINT_FILES = {
    "default": ("tabpfn-v3.5-20260909.safetensors",
                "ece4d67eadfea42eb0e610df5189bea60cb7f31073d81e9c7a019b76eacf0be3"),
    "fast":    ("tabpfn-v3.5-fast-20260909.safetensors",
                "14e0b81285bca7a6d1ef121169f27f651f76dff49cad6c0b8779d5c836018550"),
}

from huggingface_hub import hf_hub_download

for name in CHECKPOINTS_TO_DOWNLOAD:
    filename, expected_sha256 = CHECKPOINT_FILES[name]
    path = os.path.join(HF_LOCAL_DIR, filename)
    if not os.path.exists(path):
        path = hf_hub_download(repo_id=HF_REPO_ID, filename=filename, token=HF_TOKEN, local_dir=HF_LOCAL_DIR)
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            sha.update(chunk)
    status = "SHA-256 OK" if sha.hexdigest() == expected_sha256 else "SHA-256 MISMATCH: " + sha.hexdigest()
    print(f"{name}: {path} - {status}")

## Stage 0 - import libraries
In Stage 0, we define all imports necessary to run our subsequent code, including `tabpfn` itself, the local checkpoint files downloaded in Stage -1, and two small helper functions. Helpers must live in this tagged cell - anything defined in an untagged cell is not exported. If the container runs an older `tabpfn` (for example on a golden image before 5.2.5 with Python 3.9), this cell fails with a clear error message instead of failing later in `fit`/`apply`.

In [ ]:
# this definition exposes all python module imports that should be available in all subsequent commands
import json
import os
import pickle
import sys
import time

import numpy as np
import pandas as pd

import tabpfn
from tabpfn import TabPFNClassifier, TabPFNRegressor

# global constants
MODEL_DIRECTORY = "/srv/app/model/data/"
# keep in sync with CHECKPOINT_FILES in Stage -1; select with checkpoint=<name> in | fit
CHECKPOINTS = {
    "default": "tabpfn-v3.5-20260909.safetensors",       # TabPFN-3.5, ~876 MB
    "fast": "tabpfn-v3.5-fast-20260909.safetensors",     # TabPFN-3.5-Fast, ~334 MB, alpha
}
DEFAULT_CHECKPOINT = "default"

# fail fast with a clear message if the container runs an older tabpfn
if int(tabpfn.__version__.split(".")[0]) < 9:
    raise ImportError(
        f"TabPFN-3.5 needs tabpfn>=9 and Python>=3.10, found tabpfn {tabpfn.__version__} "
        f"on Python {sys.version.split()[0]}. Use a DSDL image 5.2.5 or later, or the Agentic AI image."
    )


def _get_param(param, key, default=None):
    """Read an optional SPL parameter passed after algo=, e.g. checkpoint=fast."""
    value = param.get("options", {}).get("params", {}).get(key, default)
    return value.strip('"').strip() if isinstance(value, str) else value


def _checkpoint_path(name):
    """Resolve a checkpoint name to its local file; never falls back silently."""
    if name not in CHECKPOINTS:
        raise ValueError(f"checkpoint must be one of {sorted(CHECKPOINTS)}, got {name!r}")
    path = os.path.join(MODEL_DIRECTORY, CHECKPOINTS[name])
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"checkpoint '{name}' not found at {path} - run Stage -1 with '{name}' in "
            "CHECKPOINTS_TO_DOWNLOAD or copy the file into the container"
        )
    return path

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
print("python version: " + sys.version.split()[0])
print("numpy version: " + np.__version__)
print("pandas version: " + pd.__version__)
print("tabpfn version: " + tabpfn.__version__)
for name, filename in CHECKPOINTS.items():
    print(f"checkpoint '{name}' found: {os.path.exists(os.path.join(MODEL_DIRECTORY, filename))}")

## Stage 1 - get a data sample from Splunk
To develop and test the model interactively, push a data sample from Splunk into the notebook environment. This also gives you the metadata `param` object that `fit()` and `apply()` receive from Splunk.

### Push data from Splunk
In Splunk run a search to pipe a dataset into your notebook environment. You utilize the `mode=stage` flag in the `| fit` command to do this. The search results are then accessible as a CSV file with the same model name that is defined in the `into app:<modelname>` part of the fit statement. Additionally, metadata is retrieved and accessible as a JSON file or the `param` object. In the same way you can further work with the metadata object as it is exposed in the fit and apply function definitions below in Stages 3 and 4.

```
| makeresults count=50
| eval f1=random()%100, f2=random()%100, target=if(f1>50,1,0)
| fit MLTKContainer mode=stage algo=tabpfn_model target from f1 f2 into app:tabpfn_model
```

After you run this search your dataset sample is available as a CSV file (and the accompanying metadata as JSON) inside the container to develop your model. Both file names are taken from the `into app:<modelname>` keyword ("tabpfn_model" in the example above), or set to "default" if no `into` keyword is present. This step is intended to work with a subset of your data to create your custom model.

In [ ]:
# this cell is not executed from MLTK and should only be used for staging data into the notebook environment
def stage(name):
    with open("data/"+name+".csv", 'r') as f:
        df = pd.read_csv(f)
    with open("data/"+name+".json", 'r') as f:
        param = json.load(f)
    return df, param

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
df, param = stage("tabpfn_model")

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
df

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
param

## Stage 2 - create and initialize a model
For this integration, `init()` doesn't need to set anything up - TabPFN is a frozen, pre-trained model with no per-model state to initialize before `fit()` is called.

In [ ]:
def init(df, param):
    model = {}
    return model

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
model = init(df,param)
print(model)

## Stage 3 - fit the model

Note: TabPFN's transformer weights are frozen and pre-trained - there is no gradient descent here. "Fit" caches the labeled training rows so the model can use them as in-context examples during `apply()` later. That's why `fit()` below only stores the training data, the target and feature names, and the chosen options (`task`, `checkpoint`, `ignore_pretraining_limits`), not a trained classifier object. Rows with a missing target are dropped. `fit()` returns a small info object - DSDL only writes it to the container log; the model itself is persisted by `save()` in Stage 5.

Because DSDL runs `apply()` on the training rows right after `fit()`, the results of a `| fit` search are in-sample predictions. Evaluate on held-out rows with `| apply` instead.

In [ ]:
def fit(model, df, param):
    target = param['target_variables'][0]
    features = param['feature_variables']
    task = _get_param(param, 'task', 'classification')
    if task not in ('classification', 'regression'):
        raise ValueError(f"task must be 'classification' or 'regression', got {task!r}")
    checkpoint = _get_param(param, 'checkpoint', DEFAULT_CHECKPOINT)
    _checkpoint_path(checkpoint)  # fail early if the name is wrong or the file is missing
    ignore_limits = str(_get_param(param, 'ignore_pretraining_limits', 'false')).lower() in ('true', '1', 'yes')

    data = df[features + [target]].dropna(subset=[target])
    y = data[target]
    # keep integer class labels as integers (missing values turn them into floats: 1 -> 1.0)
    if task == 'classification' and pd.api.types.is_float_dtype(y) and (y % 1 == 0).all():
        y = y.astype(int)

    model['X_train'] = data[features]
    model['y_train'] = y
    model['target_name'] = target
    model['feature_names'] = features
    model['task'] = task
    model['checkpoint'] = checkpoint
    model['ignore_pretraining_limits'] = ignore_limits
    return {"message": "training rows cached", "task": task, "checkpoint": checkpoint, "rows": len(data)}

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
print(fit(model,df,param))

## Stage 4 - apply the model

`apply()` builds a fresh TabPFN estimator from the checkpoint chosen at fit time (`checkpoint=default` or `checkpoint=fast`) and re-supplies the cached training rows on every call, rather than reusing an estimator object saved from `fit()`. A fitted estimator would embed the entire checkpoint (~0.9 GB for default, ~0.3 GB for fast) in every saved model file and tie it to one `tabpfn` version; keeping only the data keeps saved models small and lets all models share the checkpoint files on disk.

Output fields:
- classification: `predicted_<target>` plus one `probability(<target>=<class>)` field per class (AI Toolkit naming convention) - use the probability of the positive class as a ranked risk score
- regression: `predicted_<target>`

Each call writes one line with checkpoint, row counts, and runtime to the container log, which makes it easy to compare `default` and `fast`.

In [ ]:
def apply(model, df, param):
    features = model.get('feature_names') or param['feature_variables']
    X = df[features]
    target = model['target_name']
    task = model.get('task', 'classification')
    checkpoint = model.get('checkpoint', DEFAULT_CHECKPOINT)
    options = {"model_path": _checkpoint_path(checkpoint),
               "ignore_pretraining_limits": model.get('ignore_pretraining_limits', False)}
    started = time.time()

    if task == 'regression':
        reg = TabPFNRegressor(**options)
        reg.fit(model['X_train'], model['y_train'])
        result = pd.DataFrame({'predicted_' + target: reg.predict(X)})
    else:
        clf = TabPFNClassifier(**options)
        clf.fit(model['X_train'], model['y_train'])
        proba = clf.predict_proba(X)  # one forward pass; labels are derived from it
        result = pd.DataFrame({'predicted_' + target: clf.classes_[np.argmax(proba, axis=1)]})
        for i, cls in enumerate(clf.classes_):
            result[f'probability({target}={cls})'] = proba[:, i]

    print(f"tabpfn apply: checkpoint={checkpoint} task={task} train_rows={len(model['X_train'])} "
          f"rows={len(X)} seconds={time.time() - started:.2f}")
    return result

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
print(apply(model,df,param))

## Stage 5 - save the model

Only the training rows, the target and feature names, and the chosen options are pickled here, into the container's model directory (`/srv/app/model/data/<algo>_<model_name>.pkl`) - deliberately not the TabPFN weights themselves, which stay on disk as the checkpoint file and are loaded fresh by `apply()` each time.

In [ ]:
def save(model, name):
    with open(f"{MODEL_DIRECTORY}{name}.pkl", "wb") as f:
        pickle.dump(model, f)
    return model

## Stage 6 - load the model
Reconstructs the small pickled dict saved in Stage 5 - not the TabPFN weights, which `apply()` loads separately from the checkpoint file. Models saved with an older version of this notebook still work: missing options fall back to `task=classification` and `checkpoint=default`.

In [ ]:
def load(name):
    with open(f"{MODEL_DIRECTORY}{name}.pkl", "rb") as f:
        return pickle.load(f)

## Stage 7 - provide a summary of the model
Returns library versions, the available checkpoints and - for a trained model - task, checkpoint, target, features, training rows, and class counts.

In [ ]:
def summary(model=None):
    returns = {
        "version": {"tabpfn": tabpfn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
        "checkpoints_found": {name: os.path.exists(os.path.join(MODEL_DIRECTORY, f)) for name, f in CHECKPOINTS.items()},
    }
    if model and 'X_train' in model:
        task = model.get('task', 'classification')
        returns["summary"] = {
            "task": task,
            "checkpoint": model.get('checkpoint', DEFAULT_CHECKPOINT),
            "target": model['target_name'],
            "features": list(model['X_train'].columns),
            "training_rows": int(len(model['X_train'])),
        }
        if task == 'classification':
            counts = model['y_train'].value_counts()
            returns["summary"]["class_counts"] = {str(k): int(v) for k, v in counts.items()}
    return returns

**Example: predict malware-associated traffic from session metrics**

`firewall_traffic.csv` is a Splunk-collected AI Toolkit sample dataset (free to use, no citation required). `used_by_malware` (yes/no) is nicely balanced: about 59% yes and 41% no across ~99k sessions (check with `| inputlookup firewall_traffic.csv | stats count by used_by_malware`). The model predicts it from `bytes_received`, `bytes_sent`, `packets_received`, and `packets_sent`.

~99k rows is far above TabPFN's default CPU limit of 5,000 training rows, so we draw a reproducible sample (fixed seeds) of 2,000 rows and split it into a training half (~1,000 rows, below the point where TabPFN starts warning about slow CPU inference) and a held-out half. Train with the default checkpoint:

```
| inputlookup firewall_traffic.csv
| eval malware_label=if(used_by_malware=="yes",1,0)
| sample seed=123 count=2000
| sample partitions=2 fieldname=group seed=1234
| where group=0
| fit MLTKContainer algo=tabpfn_model malware_label from bytes_received bytes_sent packets_received packets_sent into app:tabpfn_firewall
```

Then apply the model to the held-out half and score it:

```
| inputlookup firewall_traffic.csv
| eval malware_label=if(used_by_malware=="yes",1,0)
| sample seed=123 count=2000
| sample partitions=2 fieldname=group seed=1234
| where group=1
| apply tabpfn_firewall
| `confusionmatrix(malware_label, predicted_malware_label)`
```

**Compare with the fast checkpoint:** train a second model with `checkpoint=fast` under its own name, run the same apply search against `tabpfn_firewall_fast`, and compare the confusion matrices and the `seconds=` values in the container log.

```
| inputlookup firewall_traffic.csv
| eval malware_label=if(used_by_malware=="yes",1,0)
| sample seed=123 count=2000
| sample partitions=2 fieldname=group seed=1234
| where group=0
| fit MLTKContainer algo=tabpfn_model checkpoint=fast malware_label from bytes_received bytes_sent packets_received packets_sent into app:tabpfn_firewall_fast
```

**Ranked risk score:** the probability of the positive class can be used directly for sorting, e.g. `| apply tabpfn_firewall | sort - "probability(malware_label=1)"`.

**Example: predict server power consumption (regression)**

`server_power.csv` is an AI Toolkit sample dataset with about 31,000 one-second measurements of a server's power draw (`ac_power`, in watts) and hardware counters such as CPU utilization, core cycles, cache references, and memory bus and disk activity. The same notebook handles this numeric target with `task=regression`; the `rename` gives the counters shorter field names. Train on one half of a reproducible 2,000-row sample:

```
| inputlookup server_power.csv
| rename "total-cpu-utilization" as cpu_util, "total-unhalted_core_cycles" as core_cycles, "total-instructions_retired" as instructions, "total-last_level_cache_references" as cache_refs, "total-memory_bus_transactions" as mem_bus, "total-disk-utilization" as disk_util
| sample seed=123 count=2000
| sample partitions=2 fieldname=group seed=1234
| where group=0
| fit MLTKContainer algo=tabpfn_model task=regression ac_power from cpu_util core_cycles instructions cache_refs mem_bus disk_util into app:tabpfn_power
```

Then apply the model to the held-out half and score it:

```
| inputlookup server_power.csv
| rename "total-cpu-utilization" as cpu_util, "total-unhalted_core_cycles" as core_cycles, "total-instructions_retired" as instructions, "total-last_level_cache_references" as cache_refs, "total-memory_bus_transactions" as mem_bus, "total-disk-utilization" as disk_util
| sample seed=123 count=2000
| sample partitions=2 fieldname=group seed=1234
| where group=1
| apply tabpfn_power
| `regressionstatistics(ac_power, predicted_ac_power)`
```
Because the measurements are only one second apart, neighboring rows in a random split are very similar, so the score is an optimistic estimate. For your own data, a time-based split (fit on an earlier period, apply to the most recent one) gives a more realistic estimate.

## End of Stages
All subsequent cells are not tagged and can be used for further freeform code.